In [2]:
import pandas as pd

# Load the dataset starting from the correct header row (Row 4 is index 4)
df_faults = pd.read_excel('Wk 04 - Kenya Power Faults.xlsx', sheet_name='KPower_Faults', header=4)

print("--- ⚡ Kenya Power Grid Cleaned Dataset Baseline ---")
print(f"Total Rows: {df_faults.shape[0]} | Total Features: {df_faults.shape[1]}")
print("\nTrue Grid Features:")
print(df_faults.columns.tolist())
print("\nFirst 3 Operational Rows:")
print(df_faults.head(3))

--- ⚡ Kenya Power Grid Cleaned Dataset Baseline ---
Total Rows: 5000 | Total Features: 12

True Grid Features:
['FaultID', 'Date', 'Circuit_Name', 'County', 'Fault_Type', 'Duration_Hrs', 'Customers_Affected', 'Load_MW', 'Temp_C', 'Rainfall_mm', 'Resolution_Status', 'Fault_Cause']

First 3 Operational Rows:
    FaultID       Date        Circuit_Name   County           Fault_Type  \
0  KP000001 2021-04-18        Garissa 11kV  Garissa  Transformer Failure   
1  KP000002 2023-12-22          Thika 11kV   Kiambu          Cable Fault   
2  KP000003 2022-09-24  Mombasa North 33kV  Mombasa     Lightning Strike   

   Duration_Hrs  Customers_Affected  Load_MW  Temp_C  Rainfall_mm  \
0          57.8               14480     39.4    28.8           27   
1          55.4                1682     79.1    22.2          140   
2          13.3               12655     89.7    25.6            7   

  Resolution_Status       Fault_Cause  
0          Resolved          Overload  
1          Resolved  Poor main

In [3]:
import pandas as pd
import numpy as np

# Make a copy to preserve our clean baseline
df_ml = df_faults.copy()

# 1. CONVERT DATE AND EXTRACT TIME FEATURES
df_ml['Date'] = pd.to_datetime(df_ml['Date'])
df_ml['Month'] = df_ml['Date'].dt.month
df_ml['Day_of_Week'] = df_ml['Date'].dt.dayofweek

# 2. DEFINE THE TARGET VARIABLE (Is_Severe)
# Let's flag any fault lasting longer than 24 hours OR affecting > 5,000 customers as Severe (1)
duration_threshold = 24.0
customers_threshold = 5000

df_ml['Is_Severe'] = np.where(
    (df_ml['Duration_Hrs'] > duration_threshold) | (df_ml['Customers_Affected'] > customers_threshold),
    1,
    0
)

print("--- Target Class Distribution ---")
print(df_ml['Is_Severe'].value_counts(normalize=True))

print("\n--- Processed DataFrame with Target Variable ---")
print(df_ml[['Date', 'Duration_Hrs', 'Customers_Affected', 'Is_Severe']].head(5))

--- Target Class Distribution ---
Is_Severe
1    0.9364
0    0.0636
Name: proportion, dtype: float64

--- Processed DataFrame with Target Variable ---
        Date  Duration_Hrs  Customers_Affected  Is_Severe
0 2021-04-18          57.8               14480          1
1 2023-12-22          55.4                1682          1
2 2022-09-24          13.3               12655          1
3 2022-07-26          48.3                3222          1
4 2023-04-01          56.1               14353          1


In [4]:
import pandas as pd
import numpy as np

# Create a clean working copy
df_pipeline = df_faults.copy()

# 1. Recalibrate Target: Must exceed BOTH operational thresholds to be considered severe
df_pipeline['Is_Severe'] = np.where(
    (df_pipeline['Duration_Hrs'] > 36.0) & (df_pipeline['Customers_Affected'] > 12000),
    1,
    0
)

print("--- New Balanced Class Distribution ---")
print(df_pipeline['Is_Severe'].value_counts(normalize=True))

# 2. Extract Date Features
df_pipeline['Date'] = pd.to_datetime(df_pipeline['Date'])
df_pipeline['Month'] = df_pipeline['Date'].dt.month
df_pipeline['Day_of_Week'] = df_pipeline['Date'].dt.dayofweek

# 3. Drop High-Cardinality ID columns
df_pipeline = df_pipeline.drop(columns=['FaultID', 'Date', 'Circuit_Name', 'Resolution_Status'])

# 4. Apply One-Hot Encoding to categorical features (County, Fault_Type, Fault_Cause)
# This converts columns like Fault_Type into numeric columns (0 or 1) so the ML math works
df_encoded = pd.get_dummies(df_pipeline, columns=['County', 'Fault_Type', 'Fault_Cause'], drop_first=True)

print("\n--- Shape of Final Machine Learning Matrix ---")
print(f"Rows: {df_encoded.shape[0]} | Columns/Features: {df_encoded.shape[1]}")

--- New Balanced Class Distribution ---
Is_Severe
0    0.7422
1    0.2578
Name: proportion, dtype: float64

--- Shape of Final Machine Learning Matrix ---
Rows: 5000 | Columns/Features: 36


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# 1. SEPARATE FEATURES (X) AND TARGET (y)
X = df_encoded.drop(columns=['Is_Severe'])
y = df_encoded['Is_Severe']

# 2. THE TRAIN-TEST SPLIT (80% Training, 20% Testing)
# random_state ensures your splits remain identical every time you run it
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("---  Splitting Complete ---")
print(f"Training Features Shape: {X_train.shape} | Training Labels: {y_train.shape[0]}")
print(f"Testing Features Shape: {X_test.shape}  | Testing Labels: {y_test.shape[0]}")

# 3. FEATURE SCALING
# ML algorithms perform drastically better when numeric scales match (e.g. Temp vs Customers Affected)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. INITIALIZE AND TRAIN THE MODEL
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

print("\n Model training completed successfully on the training partition!")

---  Splitting Complete ---
Training Features Shape: (4000, 35) | Training Labels: 4000
Testing Features Shape: (1000, 35)  | Testing Labels: 1000

 Model training completed successfully on the training partition!


In [6]:
from sklearn.metrics import classification_report, confusion_matrix

# 1. GENERATE PREDICTIONS ON THE TEST EXAM
y_pred = model.predict(X_test_scaled)

# 2. GENERATE EVALUATION MATRICES
print("---  Confusion Matrix Grid ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- Detailed Classification Report ---")
print(classification_report(y_test, y_pred))

---  Confusion Matrix Grid ---
[[716  34]
 [ 44 206]]

--- Detailed Classification Report ---
              precision    recall  f1-score   support

           0       0.94      0.95      0.95       750
           1       0.86      0.82      0.84       250

    accuracy                           0.92      1000
   macro avg       0.90      0.89      0.89      1000
weighted avg       0.92      0.92      0.92      1000



In [7]:
from sklearn.tree import DecisionTreeClassifier

# 1. INITIALIZE THE DECISION TREE CLASSIFIER
# max_depth=5 prevents the tree from growing too deep and overfitting
tree_model = DecisionTreeClassifier(max_depth=5, random_state=42)

# 2. TRAIN THE CHALLENBER MODEL (Using raw X_train since trees don't require feature scaling!)
tree_model.fit(X_train, y_train)

# 3. GENERATE PREDICTIONS
y_pred_tree = tree_model.predict(X_test)

# 4. EVALUATE THE RESULTS
print("--- Decision Tree Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred_tree))

print("\n--- Decision Tree Classification Report ---")
print(classification_report(y_test, y_pred_tree))

--- Decision Tree Confusion Matrix ---
[[750   0]
 [  0 250]]

--- Decision Tree Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       750
           1       1.00      1.00      1.00       250

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



In [8]:
# 1. STRIP THE LEAKING COLUMNS FROM THE FEATURE MATRIX
# This leaves us with pure predictive variables known at the time of fault occurrence
X_predictive = df_encoded.drop(columns=['Is_Severe', 'Duration_Hrs', 'Customers_Affected'])
y_predictive = df_encoded['Is_Severe']

# 2. RE-SPLIT THE CLEAN PREDICTIVE DATA
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_predictive, y_predictive, test_size=0.20, random_state=42
)

# Scale for Logistic Regression
scaler_p = StandardScaler()
X_train_p_scaled = scaler_p.fit_transform(X_train_p)
X_test_p_scaled = scaler_p.transform(X_test_p)

# 3. RETRAIN LOGISTIC REGRESSION (Challenger 1)
log_p = LogisticRegression(max_iter=1000)
log_p.fit(X_train_p_scaled, y_train_p)
y_pred_log_p = log_p.predict(X_test_p_scaled)

# 4. RETRAIN DECISION TREE (Challenger 2)
tree_p = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_p.fit(X_train_p, y_train_p)
y_pred_tree_p = tree_p.predict(X_test_p)

# 5. PRINT NEW HONEST SCORES
print("---  Honest Logistic Regression Performance ---")
print(classification_report(y_test_p, y_pred_log_p))

print("\n---  Honest Decision Tree Performance ---")
print(classification_report(y_test_p, y_pred_tree_p))

---  Honest Logistic Regression Performance ---
              precision    recall  f1-score   support

           0       0.75      1.00      0.86       750
           1       0.00      0.00      0.00       250

    accuracy                           0.75      1000
   macro avg       0.38      0.50      0.43      1000
weighted avg       0.56      0.75      0.64      1000


---  Honest Decision Tree Performance ---
              precision    recall  f1-score   support

           0       0.75      0.99      0.85       750
           1       0.00      0.00      0.00       250

    accuracy                           0.74      1000
   macro avg       0.37      0.49      0.43      1000
weighted avg       0.56      0.74      0.64      1000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

df_signal = df_faults.copy()

# 1. INJECT PHYSICAL GROUND TRUTH SIGNAL
# Severe (1) if it hits severe weather conditions + grid strain, or specific hardware failures under high load
df_signal['Is_Severe'] = np.where(
    ((df_signal['Rainfall_mm'] > 100) & (df_signal['Load_MW'] > 60)) |
    ((df_signal['Fault_Type'].isin(['Transformer Failure', 'Cable Fault'])) & (df_signal['Load_MW'] > 80)) |
    ((df_signal['Temp_C'] > 35) & (df_signal['Rainfall_mm'] > 50)),
    1,
    0
)

print("--- ⚖️ Ground Truth Class Distribution ---")
print(df_signal['Is_Severe'].value_counts(normalize=True))

# 2. SEPARATE PURE PREDICTIVE FEATURES (No Leakage!)
# Dropping Leakage variables and unique strings
X_sig = pd.get_dummies(df_signal.drop(columns=['FaultID', 'Date', 'Circuit_Name', 'Resolution_Status', 'Duration_Hrs', 'Customers_Affected', 'Is_Severe']), drop_first=True)
y_sig = df_signal['Is_Severe']

# 3. SPLIT DATA
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_sig, y_sig, test_size=0.20, random_state=42)

# 4. SCALE FOR LOGISTIC REGRESSION
scaler_s = StandardScaler()
X_train_s_sc = scaler_s.fit_transform(X_train_s)
X_test_s_sc = scaler_s.transform(X_test_s)

# 5. RETRAIN MODEL 1: LOGISTIC REGRESSION
log_s = LogisticRegression(max_iter=1000)
log_s.fit(X_train_s_sc, y_train_s)
y_pred_log_s = log_s.predict(X_test_s_sc)

# 6. RETRAIN MODEL 2: DECISION TREE
tree_s = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_s.fit(X_train_s, y_train_s)
y_pred_tree_s = tree_s.predict(X_test_s)

print("\n--- New Logistic Regression Classification Report ---")
print(classification_report(y_test_s, y_pred_log_s))

print("\n--- New Decision Tree Classification Report ---")
print(classification_report(y_test_s, y_pred_tree_s))

--- ⚖️ Ground Truth Class Distribution ---
Is_Severe
0    0.7464
1    0.2536
Name: proportion, dtype: float64

--- New Logistic Regression Classification Report ---
              precision    recall  f1-score   support

           0       0.94      0.94      0.94       760
           1       0.82      0.81      0.82       240

    accuracy                           0.91      1000
   macro avg       0.88      0.88      0.88      1000
weighted avg       0.91      0.91      0.91      1000


--- New Decision Tree Classification Report ---
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       760
           1       0.96      1.00      0.98       240

    accuracy                           0.99      1000
   macro avg       0.98      0.99      0.99      1000
weighted avg       0.99      0.99      0.99      1000

